# PRAA-FL on Colab (T4)
Run top to bottom. Real PathMNIST needs internet (Colab has it). **Not executed in the build sandbox** - the commands are the ones documented in `docs/EXPERIMENTS.md`. Test split is only read in the last cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
# 1) upload/unzip the project ZIP, then:
%cd /content/PRAA-FL
!pip install -q -r requirements.txt
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 2) tests (offline, synthetic data) + preserved-evidence check
!python -m pytest -q tests
!python scripts/verify_preserved.py

In [ ]:
# 3) put the ORIGINAL Phase 3 model next to the preserved evidence and verify its SHA-256
!cp /content/drive/MyDrive/phase3/final_model.pt results/phase3/final_model.pt
!python scripts/verify_preserved.py --phase3-model results/phase3/final_model.pt

In [ ]:
# 4) measure the real step time on this GPU and update the estimate
!python estimate_compute.py --benchmark --output results/compute_estimate_gpu.json

In [ ]:
# 5) cheap controlled pilot from the preserved model (30 rounds per arm), then tune tau on VALIDATION
OUT = "/content/drive/MyDrive/praa_runs"
!python run_experiment.py --method flame --config configs/pilot_continue_from_phase3.json --output-dir {OUT}/flame_pilot
!python run_phase7.py --config configs/pilot_continue_from_phase3.json --output-dir {OUT}/praa_pilot
!python run_phase8.py evaluate --run-dir {OUT}/flame_pilot --split val
!python run_phase8.py evaluate --run-dir {OUT}/praa_pilot --split val
!python run_phase8.py tune-tau --run-dir {OUT}/praa_pilot --taus 0 0.5 1 2 4 8
# after a disconnect, add --resume to the same run command

In [ ]:
# 6) full runs (hours; resumable). Replace TAU with the value chosen on validation (1.0 = documented formula)
TAU = 1.0
!python run_experiment.py --method fedavg --output-dir {OUT}/fedavg
!python run_experiment.py --method flame --output-dir {OUT}/flame
!python run_phase7.py --set praa.tau={TAU} --output-dir {OUT}/praa

In [ ]:
# 7) validation comparison, then the FINAL test evaluation (once)
for m in ["fedavg","flame","praa"]:
    !python run_phase8.py evaluate --run-dir {OUT}/{m} --split val
!python run_phase8.py evaluate-original --split val --output results/phase8/eval_original_val.json
!python run_phase8.py compare --run FedAvg={OUT}/fedavg --run FLAME={OUT}/flame --run PRAA={OUT}/praa --original results/phase8/eval_original_val.json --split val
# FINAL (test split) - only after freezing every setting on validation:
# for m in ["fedavg","flame","praa"]:
#     !python run_phase8.py evaluate --run-dir {OUT}/{m} --split test --final
# !python run_phase8.py evaluate-original --split test --final --output results/phase8/eval_original_test.json
# !python run_phase8.py compare --run FedAvg={OUT}/fedavg --run FLAME={OUT}/flame --run PRAA={OUT}/praa --original results/phase8/eval_original_test.json --split test --output-dir results/phase8_test